# awkward 的数据类型 ― 从类型读出结构

## 本章要学什么

awkward 的数组能处理有层次的数据和长度会变的数据，这些结构都会体现在 `type` 属性里，所以读懂类型，就能知道数组内部长什么样。这一章造出各种数据，确认 `type` 会显示成什么样子。

* 一维数组，以及固定长轴与可变长轴的表示
* 值的缺失(`?int64`)与列表本身的缺失(`option[...]`)
* 类型混杂的数组(union)
* 结构体(记录)与元组

读懂 `type`，是了解 awkward 内部结构的第一步。

In [1]:
import numpy as np
import awkward as ak
from IPython.core.magic import register_line_magic
import html

## 取得数据类型

Awkward Array 可以用 `type` 和 `repr(type)` 这两种形式显示类型：前者是简写，后者是内部构造的详细写法。两种写法描述的是同一个数组，只是详略不同。

In [2]:
a = ak.Array([[1, 2, 3], [4, 5]])
print(a.type)
print(repr(a.type))

2 * var * int64
ArrayType(ListType(NumpyType('int64')), 2, None)


这里能同时得到两种写法：

* `2 * var * int64` 这样的简写
* `ArrayType(ListType(...))` 这样的详细写法

简写把结构压缩成一行，适合日常查看；详细写法则把每一层都展开，排查问题时更有用。

为了便于查看，这里定义 `%t` 魔法命令，把 Awkward Array 的值和类型一起显示出来。

In [3]:
@register_line_magic
def t(line):
    arr = get_ipython().ev(line)
    html_str = html.unescape(arr._repr_mimebundle_()['text/html'])
    type_str = repr(arr.type)
    html_str = html_str.replace("</pre>", f"\ntype: {type_str}").replace("<pre>", "")
    print(html_str)

## 一维数组

一维整数数组的类型就简单地表示成“元素数 × 基本类型”。开头那个数表示轴的长度，后面的 `int64` 表示元素类型。

In [4]:
%t ak.Array([1, 2, 3, 4])

[1,
 2,
 3,
 4]
---
backend: cpu
nbytes: 32 B
type: 4 * int64
type: ArrayType(NumpyType('int64'), 4, None)


## 元素缺失(Option 型)

含有 `None` 时，该元素就用 `?int64` 这样的 OptionType 表示。`?` 的意思是“可能是这个类型，也可能是缺失”，后面照旧写着正常情况下的类型。

In [5]:
%t ak.Array([1, None, 3, 4])

[1,
 None,
 3,
 4]
------
backend: cpu
nbytes: 56 B
type: 4 * ?int64
type: ArrayType(OptionType(NumpyType('int64')), 4, None)


## 二维数组(规则轴)

传入 NumPy 的规则(固定长)二维数组时，它会被表示成 `regular` 轴，如 `4 * 3 * int32`。；两个长度都固定，可以直接写成两个数相乘的形式，从 NumPy 转过来的数组经常是这种类型。

In [6]:
%t ak.Array(np.random.randint(0, 10, (4, 3)))

[[4, 3, 1],
 [4, 0, 5],
 [1, 7, 6],
 [1, 8, 5]]
-----------
backend: cpu
nbytes: 48 B
type: 4 * 3 * int32
type: ArrayType(RegularType(NumpyType('int32'), 3), 4, None)


## 二维数组(非规则轴)

各列表长度不一致的轴会用 `var` 表示成可变长；前面的 `3 *` 仍然是固定长的外层维数，`var` 代替了内层那个固定的数，长度改由数组自己记录。

In [7]:
%t ak.Array([[1, 2, 3], [4, 5], [6]])

[[1, 2, 3],
 [4, 5],
 [6]]
-----------
backend: cpu
nbytes: 80 B
type: 3 * var * int64
type: ArrayType(ListType(NumpyType('int64')), 3, None)


## 多个非规则轴

有多个非规则轴时，会用 `var * var` 这样的形式分层地表示出来。每一层对应一级嵌套的列表，从右往左读就是从内到外。

In [8]:
%t ak.Array([[[1], [2], [3, 3]], [[4, 5]], [[6, 7]]])

[[[1], [2], [3, 3]],
 [[4, 5]],
 [[6, 7]]]
--------------------
backend: cpu
nbytes: 144 B
type: 3 * var * var * int64
type: ArrayType(ListType(ListType(NumpyType('int64'))), 3, None)


## 非规则轴 + 元素缺失

可变长列表里含有 `None` 时，元素的类型就变成 `?int64`；可变长的部分照旧用 `var`，只是元素类型前面多了一个 `?`。

In [9]:
%t ak.Array([[1, None, 3], [4, 5], [6]])

[[1, None, 3],
 [4, 5],
 [6]]
--------------
backend: cpu
nbytes: 120 B
type: 3 * var * ?int64
type: ArrayType(ListType(OptionType(NumpyType('int64'))), 3, None)


## 非规则轴 + 列表缺失

列表本身就是 `None` 时，外层的类型变成 `option[var * int64]`；与元素缺失不同，这里的 `option` 包住的是整个列表，而不是列表里的某个元素。

In [10]:
%t ak.Array([[1, 2, 3], None, [4, 5], [6]])

[[1, 2, 3],
 None,
 [4, 5],
 [6]]
-----------
backend: cpu
nbytes: 112 B
type: 4 * option[var * int64]
type: ArrayType(OptionType(ListType(NumpyType('int64'))), 4, None)


## 非规则轴 + 列表缺失 + 元素缺失

这是列表缺失与元素缺失同时存在时类型的写法：外层的 `option[...]` 管列表，里面的 `?int64` 管元素。两层缺失各管各的，互不干扰。

In [11]:
%t ak.Array([[1, None, 3], None, [4, 5], [6]])

[[1, None, 3],
 None,
 [4, 5],
 [6]]
--------------
backend: cpu
nbytes: 152 B
type: 4 * option[var * ?int64]
type: ArrayType(OptionType(ListType(OptionType(NumpyType('int64')))), 4, None)


## Union 型

像整数与字符串这样含有异种数据的数组，类型是 `union[...]`。每个元素都带着自己的类型标记，取出时才能还原成原来的类型。

In [12]:
%t ak.Array([1, 'x', 2, 'y'])

[1,
 'x',
 2,
 'y']
-----
backend: cpu
nbytes: 78 B
type: 4 * union[
    int64,
    string
]
type: ArrayType(UnionType([NumpyType('int64'), ListType(NumpyType('uint8', parameters={'__array__': 'char'}), parameters={'__array__': 'string'})]), 4, None)


## 结构体型

Awkward Array 也能处理带有 `{x: ..., y: ...}` 这类字段的结构体型(RecordType)，可以把它看成一列有名字的字段，每个字段各自是一个数组。

In [13]:
%t ak.Array({'x':[1, 2, 3], 'y':[4, 5, 6]})

[{x: 1, y: 4},
 {x: 2, y: 5},
 {x: 3, y: 6}]
--------------
backend: cpu
nbytes: 48 B
type: 3 * {
    x: int64,
    y: int64
}
type: ArrayType(RecordType([NumpyType('int64'), NumpyType('int64')], ['x', 'y']), 3, None)


## 结构体型 + 缺失字段

同一个结构体里只让某个字段缺失也是可以的：该字段的类型会表示成 `?int64`，而其他字段仍然照常取值，长度也不变。

In [14]:
%t ak.Array([{'x': 1, 'y': 4}, {'x': 2, 'y': 5}, {'y': 6}])

[{x: 1, y: 4},
 {x: 2, y: 5},
 {x: None, y: 6}]
-----------------
backend: cpu
nbytes: 64 B
type: 3 * {
    x: ?int64,
    y: int64
}
type: ArrayType(RecordType([OptionType(NumpyType('int64')), NumpyType('int64')], ['x', 'y']), 3, None)


## 结构体型 + 含列表的字段

字段本身就带有列表时，会写成 `y: var * int64` 这样。字段的类型同样是完整的数组类型，因此可以继续往里嵌套。

In [15]:
%t ak.Array([{'x': 2, 'y': [4, 5]}, {'x': 3, 'y': [5]}, {'x':4, 'y': [6, 7, 8]}])

[{x: 2, y: [4, 5]},
 {x: 3, y: [5]},
 {x: 4, y: [6, 7, 8]}]
----------------------
backend: cpu
nbytes: 104 B
type: 3 * {
    x: int64,
    y: var * int64
}
type: ArrayType(RecordType([NumpyType('int64'), ListType(NumpyType('int64'))], ['x', 'y']), 3, None)


## 元组型

没有字段名的结构体当作元组型处理，形如 `(int64, string)`。它与结构体的区别只是字段没有名字，位置就是身份。

In [16]:
%t ak.zip([[1, 2, 3], ['x', 'y', 'z']])

[(1, 'x'),
 (2, 'y'),
 (3, 'z')]
----------
backend: cpu
nbytes: 59 B
type: 3 * (
    int64,
    string
)
type: ArrayType(RecordType([NumpyType('int64'), ListType(NumpyType('uint8', parameters={'__array__': 'char'}), parameters={'__array__': 'string'})], None), 3, None)


## 小结

能读懂 `type` 的写法，数组的结构就一目了然了。下面把这一章出现的类型整理成一张表。

| 数据 | 类型的写法 |
|:--|:--|
| 一维整数 | `4 * int64` |
| 含值缺失 | `4 * ?int64` |
| 固定长二维 | `4 * 3 * int32` |
| 长度会变的轴 | `3 * var * int64` |
| 嵌套的可变长 | `var * var * int64` |
| 列表缺失 | `option[var * int64]` |
| 类型混在 | `union[int64, string]` |
| 结构体 | `{x: int64, y: int64}` |
| 元组 | `(int64, string)` |

其中 `?` 表示值的缺失，`option[...]` 表示列表本身的缺失，而 `var` 表示长度会变的轴。